# Commercial models through their APIs

Generates the replies of GPT-6 Luna and Gemini 3.8 Flash for `runs/run.py --import` (see `runs/README.md`).

1. No GPU is needed (Accelerator: None). Turn Internet on.
2. Add-ons → Secrets: attach `OPENAI_API_KEY` and `GEMINI_API_KEY` to this notebook. The keys are read from the
   secrets and never printed.
3. Upload `generate_api.py` and the requests file `requests_<RUN>.jsonl` (on Kaggle as a dataset), and set `RUN`
   and the budgets below. Every call stops before its budget (USD) would be exceeded.
4. Run all cells, then download `<RUN>_outputs.tar.gz`, which holds the `replies_*_<RUN>.jsonl` files and
   their `.meta.json` files.


In [ ]:
import glob, os

RUN = "probe"  # the requests file is requests_<RUN>.jsonl

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(name))
    if not hits:
        raise FileNotFoundError(f"upload {name} first")
    return hits[0]

SCRIPT = find("generate_api.py")
REQUESTS = find(f"requests_{RUN}.jsonl")
print(SCRIPT, REQUESTS, sep="\n")


In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ("OPENAI_API_KEY", "GEMINI_API_KEY"):
    try:
        os.environ[name] = secrets.get_secret(name)
        print(name, "loaded")
    except Exception as e:
        print(name, "not available:", type(e).__name__)


In [ ]:
!python {SCRIPT} gpt-6-luna {REQUESTS} replies_gpt-6-luna_{RUN}.jsonl --budget 0.30


In [ ]:
!python {SCRIPT} gemini-3.8-flash {REQUESTS} replies_gemini-3.8-flash_{RUN}.jsonl --budget 2.00


In [ ]:
!ls -la replies_*_{RUN}.jsonl*
!tar czf {RUN}_outputs.tar.gz replies_*_{RUN}.jsonl*
